# ETL

*Extracción, transformación y carga: de las líneas de pedido a la tabla del modelo*

In [ ]:
# Librerías y funciones compartidas del libro (common.py)
import plotly.express as px

from common import CATEGORY_COLORS, kpis, load_data, load_mensual, show, style_fig, tabla, usd

# df: una fila por línea de pedido · m: ventas por categoría × región × mes (salidas del ETL)
df = load_data()
m = load_mensual()
# Meses en que una categoría no vendió nada en una región (quedan con 0, no desaparecen)
ceros = int((m.Sales == 0).sum())
# Cifras clave de la calidad de los datos
kpis([
    ("Líneas de pedido", f"{len(df):,}", "2014-01-03 a 2017-12-30"),
    ("Valores faltantes", f"{int(df.isna().sum().sum())}", "en las 21 columnas"),
    ("Filas duplicadas", f"{int(df.duplicated().sum())}", "registros repetidos"),
    ("Tabla mensual", f"{len(m)} filas", f"12 series × 48 meses · {ceros} meses sin ventas"),
])

## Código del ETL (`data/generate_data.py`)

```python
# --- Extracción
df = pd.read_csv("data/raw/Sample - Superstore.csv", encoding="latin1")

# --- Transformación
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%m/%d/%Y")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], format="%m/%d/%Y")
df = df.rename(columns={"Order Date": "Order_Date"}).sort_values("Order_Date")

# Tabla mensual: ventas por categoría × región × mes (un mes sin pedidos queda en 0)
mensual = (df.groupby(["Category", "Region", pd.Grouper(key="Order_Date", freq="MS")])
             .Sales.sum().reindex(todas_las_combinaciones, fill_value=0))

# --- Carga
df.to_csv("data/superstore_transformado.csv", index=False)      # EDA
mensual.to_csv("data/ventas_mensuales.csv", index=False)       # modelo
```

Las fechas vienen como texto mes/día/año y se convierten a tipo fecha. La tabla mensual incluye todas
las combinaciones de categoría, región y mes: si en un mes no hubo pedidos, la venta queda en 0 en vez
de desaparecer, para que cada serie tenga sus 48 meses completos.

## Tabla mensual (muestra de 2017)

In [ ]:
# Primeras filas de 2017 de la tabla mensual, con la fecha como año-mes y las ventas en USD
muestra = m[m.Mes.dt.year == 2017].head(8).assign(Mes=lambda d: d.Mes.dt.strftime("%Y-%m"),
                                                   Sales=lambda d: d.Sales.map(lambda v: usd(v, 2)))
# Columnas con nombres en español para mostrarlas
tabla(muestra.rename(columns={"Category": "Categoría", "Region": "Región", "Sales": "Ventas (USD)"}))

Cada fila es una categoría en una región durante un mes. Es la unidad que el modelo proyecta: 3
categorías × 4 regiones = 12 series.

## Ventas mensuales por categoría (resultado de la tabla mensual)

In [ ]:
# Ventas totales por mes y categoría (suma de las 4 regiones)
serie = m.groupby(["Mes", "Category"], as_index=False).Sales.sum()
# Una línea por categoría, con su color fijo
fig = px.line(serie, x="Mes", y="Sales", color="Category", color_discrete_map=CATEGORY_COLORS,
              labels={"Sales": "Ventas (USD)", "Mes": "", "Category": "Categoría"})
fig.update_traces(line_width=2)
show(style_fig(fig, 330))

Las tres categorías repiten el mismo patrón cada año —ventas bajas a comienzos de año y picos de
septiembre a diciembre— y crecen desde 2016. Ese patrón repetible es lo que el modelo aprende para
proyectar.